In [ ]:
# Example 1: Top-level public API surface check.
import langchain_fireworks

expected_names = ["ChatFireworks", "Fireworks", "FireworksEmbeddings"]
found = {name: hasattr(langchain_fireworks, name) for name in expected_names}
for name, present in found.items():
    print(f"  langchain_fireworks.{name}: {'OK' if present else 'MISSING'}")

# At least the chat model should be exposed at top level.
assert found["ChatFireworks"], "ChatFireworks should be exposed at top level"


In [ ]:
# Example 2: Construct ChatFireworks with a dummy API key (no network call).
import os

from langchain_fireworks import ChatFireworks

os.environ.setdefault("FIREWORKS_API_KEY", "dummy-key-for-tests")

chat = ChatFireworks(
    model="accounts/fireworks/models/llama-v3-8b-instruct",
    temperature=0.0,
    max_tokens=16,
)

print(f"type(chat) = {type(chat).__name__}")
print(f"chat.model_name = {chat.model_name}")
print(f"chat.temperature = {chat.temperature}")
assert chat.model_name == "accounts/fireworks/models/llama-v3-8b-instruct"
assert chat.temperature == 0.0


In [ ]:
# Example 3: Construct the Fireworks completions LLM and inspect the params that identify it.
# In langchain-fireworks 1.4.x the completions LLM does NOT override `_identifying_params` (the
# LangChain base returns `lc_attributes`, empty for this class). Its identity is carried by the
# public `default_params` (the request payload) and by `_get_ls_params()` (the LangSmith tracing
# contract). `ChatFireworks` is the class in this package that overrides `_identifying_params`.
import os

from langchain_fireworks import ChatFireworks, Fireworks

os.environ.setdefault("FIREWORKS_API_KEY", "dummy-key-for-tests")

model = "accounts/fireworks/models/llama-v3-8b-instruct"
llm = Fireworks(model=model, temperature=0.2, max_tokens=8)

print(f"type(llm) = {type(llm).__name__}")
print(f"llm._llm_type = {llm._llm_type}")
print(f"llm.default_params = {llm.default_params}")
ls_params = dict(llm._get_ls_params())
print(f"llm._get_ls_params() = {ls_params}")

assert llm._llm_type == "fireworks"
assert llm.default_params["model"] == model
assert llm.default_params["temperature"] == 0.2
assert llm.default_params["max_tokens"] == 8
assert ls_params["ls_model_name"] == model
assert (ls_params["ls_provider"], ls_params["ls_model_type"]) == ("fireworks", "llm")

# The chat model is this package's carrier of the `_identifying_params` contract.
chat_params = ChatFireworks(model=model, temperature=0.0)._identifying_params
print(f"ChatFireworks identifying params = {sorted(chat_params)}")
assert chat_params["model_name"] == model

In [ ]:
# Example 4: Construct FireworksEmbeddings and inspect default model name.
import os

from langchain_fireworks import FireworksEmbeddings

os.environ.setdefault("FIREWORKS_API_KEY", "dummy-key-for-tests")

emb = FireworksEmbeddings(model="nomic-ai/nomic-embed-text-v1.5")

print(f"type(emb) = {type(emb).__name__}")
print(f"emb.model = {emb.model}")
# Should be a BaseModel-style object with a method 'embed_documents'.
assert callable(getattr(emb, "embed_documents", None))
assert callable(getattr(emb, "embed_query", None))


In [ ]:
# Example 5: ChatFireworks integrates with langchain_core message types and prompt templates.
import os

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

from langchain_fireworks import ChatFireworks

os.environ.setdefault("FIREWORKS_API_KEY", "dummy-key-for-tests")

chat = ChatFireworks(model="accounts/fireworks/models/llama-v3-8b-instruct")

# Bind a couple of runtime params; check the resulting Runnable has expected interface.
bound = chat.bind(stop=["\n\n"])
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are concise."),
    ("human", "{question}"),
])
chain = prompt | bound

msgs = [SystemMessage(content="hi"), HumanMessage(content="hello")]
print(f"chain type = {type(chain).__name__}")
print(f"msgs roles = {[m.type for m in msgs]}")
assert callable(getattr(chain, "invoke", None))
assert hasattr(chat, "bind_tools")
